# Model 2 — Second real-outcome source: Dutch RDW open data (public domain)  (experiments only)

**Why a second source.** The UK MOT pilot (notebook 04) has true odometer readings but registry attributes only of make/model/fuel/age. The Dutch RDW publishes, **for every plate**,
(a) the vehicle register (make, model, first-admission date, engine size, mass, ...) and (b) every periodic inspection (APK) with the **defects found, by regulation article**.
That is the closest open analogue of "registry fields in, real inspection outcome out" and it covers a whole national fleet (24.7 M periodic inspections).

**What the outcome is / is not.** Same caveat as the MOT: a *defect found at the periodic inspection*, grouped by the regulation article, **not** a breakdown and **not** a future failure. Systems used:
* **brakes**: articles 5.\*.31–5.\*.40 (brake components, fluid, braking performance, parking brake) + ABS/electronic-brake warning (RA6, RA7) + brake-wear advisory (AC5);
* **electrical**: lighting and signalling 5.\*.51–5.\*.65, battery and wiring 5.\*.12 / 5.\*.12a (incl. EV drive-line insulation), horn/eCall 5.\*.71;
* **engine-related**: exhaust emissions and noise 5.\*.11 / 5.\*.11a, fuel system 5.\*.9, gas tanks 5.\*.10 (a, b), engine mounts 5.\*.13. **Not engine health**: it only contains what an inspector checks.

**Data pulled** (`ml/src/rdw_nl_sample.py`): passenger cars whose plate ends in the letter K (the last plate character cycles fastest, so this is ~1/20 of the fleet spread evenly over registration years); all their light-vehicle inspections and defects.
**There is no odometer in the open RDW data** (only a plausibility judgement); this matches Israel, where mileage will also be an estimate.

In [1]:
import sys, json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.options.display.width = 170; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import INTERIM_DIR
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
try:
    display
except NameError:
    display = print
load = lambda n: pd.DataFrame(json.loads((INTERIM_DIR / n).read_text(encoding="utf-8")))
reg, fuel, tests, defects, cat = load("nl_registry_K.json"), load("nl_fuel_K.json"), load("nl_tests_K.json"), load("nl_defects_K.json"), load("nl_defect_catalogue.json")
print({k: len(v) for k, v in dict(registry=reg, fuel=fuel, tests=tests, defects=defects, catalogue=cat).items()})

{'registry': 507244, 'fuel': 828362, 'tests': 979187, 'defects': 891814, 'catalogue': 1002}


## 1. Defect-to-system mapping (from the official defect catalogue) and test table

In [2]:
def system_of(article: str):
    a = str(article)
    m = re.match(r"^5\.\*\.(\d+)([a-zA-Z]?)$", a)
    if m:
        n, suf = int(m.group(1)), m.group(2).lower()
        if 31 <= n <= 40: return "brakes"
        if 51 <= n <= 65 or (n == 12) or n == 71: return "electrical"
        if n in (9, 10, 11, 13): return "engine_emissions"
    if a in ("RA6", "RA7", "AC5"): return "brakes"
    return "other"
cat["system"] = cat.gebrek_artikel_nummer.map(system_of)
id2sys = cat.drop_duplicates("gebrek_identificatie").set_index("gebrek_identificatie").system
print(cat.groupby("system").gebrek_identificatie.nunique().to_dict(), "(distinct defect ids in the catalogue)")
for s in ["brakes", "electrical", "engine_emissions"]:
    print(s, "->", cat[cat.system == s].gebrek_omschrijving.drop_duplicates().head(6).tolist())

defects["system"] = defects.gebrek_identificatie.map(id2sys).fillna("other")
defects["n"] = pd.to_numeric(defects.aantal_gebreken_geconstateerd, errors="coerce").fillna(1)
print("defect rows by system:", defects.groupby("system").n.sum().astype(int).to_dict())

{'brakes': 134, 'electrical': 194, 'engine_emissions': 83, 'other': 591} (distinct defect ids in the catalogue)
brakes -> ['Borging remonderdelen ontbreekt', 'Hoofdcilinder werkt niet juist/zit los, rem werkt nog', 'Hoofdcilinder zit los', 'Onderdeel reminrichting beschadigd, gescheurd of gebroken', 'Onderdeel reminrichting niet juist bevestigd', 'Onderdeel reminrichting ondeugdelijk bevestigd']
electrical -> ['Aantal lichten niet juist volgens voorschrift', 'Plaats licht(en) wijkt (wijken) af', 'Voorgeschreven licht(en) ontbreekt (ontbreken)', 'Kleur lichten wijkt af/niet toegestaan', 'Verpl. licht werkt niet (goed) (geen richtingaanw./remlicht)', 'Lichten niet van gelijke grootte/sterkte/kleur']
engine_emissions -> ['Brandstofleidingen gescheurd', 'Brandstoflekkage', 'Brandstoflekkage; ernstig', 'Brandstofsysteem onveilig', 'Brandstofsysteem in ernstige mate onveilig', 'Controle alle brandstoffen niet mogelijk, niet gemeld']


defect rows by system: {'brakes': 112051, 'electrical': 321414, 'engine_emissions': 34086, 'other': 732344}


In [3]:
t = tests[tests.soort_melding_ki_omschrijving == "periodieke controle"].copy()
t["date"] = pd.to_datetime(t.meld_datum_door_keuringsinstantie, format="%Y%m%d", errors="coerce")
t = t.dropna(subset=["date"]).drop_duplicates(["kenteken", "date"]).sort_values(["kenteken", "date"])
# re-inspections after a failed test would be recorded as new periodic records a few days later: drop a record when the same plate was tested in the previous 60 days
t["gap"] = t.groupby("kenteken").date.diff().dt.days
n0 = len(t); t = t[~(t.gap < 60)].copy()
print(f"periodic inspection records: {n0:,}; after removing probable re-inspections (<60 days after the previous one): {len(t):,}")
defects["date"] = pd.to_datetime(defects.meld_datum_door_keuringsinstantie, format="%Y%m%d", errors="coerce")
dd = defects.groupby(["kenteken", "date", "system"]).n.sum().unstack(fill_value=0)
t = t.merge(dd.reset_index(), on=["kenteken", "date"], how="left")
for s in ["brakes", "electrical", "engine_emissions", "other"]:
    t[s] = (t.get(s).fillna(0) > 0).astype(int)
t["any_defect"] = (t[["brakes", "electrical", "engine_emissions", "other"]].sum(axis=1) > 0).astype(int)

reg = reg.drop_duplicates("kenteken")
reg["first"] = pd.to_datetime(reg.datum_eerste_toelating, format="%Y%m%d", errors="coerce")
fu = fuel.copy(); fu["vn"] = pd.to_numeric(fu.brandstof_volgnummer, errors="coerce")
elec_plates = set(fu[fu.brandstof_omschrijving.str.lower().isin(["elektriciteit"])].kenteken)
nfuel = fu.groupby("kenteken").size()
prim = fu.sort_values("vn").drop_duplicates("kenteken").set_index("kenteken").brandstof_omschrijving.str.lower()
FMAP = {"benzine": "petrol", "diesel": "diesel", "lpg": "lpg_cng", "cng": "lpg_cng", "elektriciteit": "electric", "alcohol": "other", "waterstof": "other"}
reg["fuel"] = reg.kenteken.map(prim).map(FMAP).fillna("other")
hyb = reg.kenteken.map(nfuel).fillna(1).gt(1) & reg.kenteken.isin(elec_plates) & (reg.fuel != "electric")
reg.loc[hyb, "fuel"] = "hybrid"
d = t.merge(reg[["kenteken", "merk", "handelsbenaming", "first", "fuel", "cilinderinhoud", "massa_rijklaar", "catalogusprijs", "taxi_indicator", "export_indicator"]], on="kenteken", how="inner")
d["age_year"] = d.date.dt.year - d["first"].dt.year            # the Israeli registry has year of manufacture/first use, so integer years is what we would have there
d["age"] = (d.date - d["first"]).dt.days / 365.25
d = d[d.age_year.between(3, 30) & d.fuel.isin(["petrol", "diesel", "hybrid", "electric", "lpg_cng"]) & d.merk.notna() & (d.taxi_indicator != "Ja")]
d["make"] = d.merk.str.title(); d["model"] = d.make + " " + d.handelsbenaming.fillna("?").str.title()
d["make"] = d.make.where(d.make.isin(d.make.value_counts().head(60).index), "OTHER")
d["model"] = d.model.where(d.model.isin(d.model.value_counts().head(400).index), "OTHER")
for c in ["cilinderinhoud", "massa_rijklaar", "catalogusprijs"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
print(f"modelling table: {len(d):,} inspections of {d.kenteken.nunique():,} cars (age 3-30 years, known fuel, no taxis)")
print("first-inspection age: the NL APK for cars starts at age 4, then 6, 8, then yearly; ages seen:", d.age_year.value_counts().sort_index().head(8).to_dict())
TARGETS = {"brakes": "brakes", "electrical (lighting, battery, wiring)": "electrical", "engine-related (emissions, fuel, mounts)": "engine_emissions", "any defect": "any_defect"}
print("prevalence per inspection:", {k: round(float(d[v].mean()), 4) for k, v in TARGETS.items()})
print(d.groupby("fuel")[["brakes", "electrical", "engine_emissions"]].mean().round(4).join(d.fuel.value_counts().rename("n")))

periodic inspection records: 973,533; after removing probable re-inspections (<60 days after the previous one): 954,177


modelling table: 812,021 inspections of 371,250 cars (age 3-30 years, known fuel, no taxis)
first-inspection age: the NL APK for cars starts at age 4, then 6, 8, then yearly; ages seen: {3: 12431, 4: 76837, 5: 26541, 6: 76646, 7: 35180, 8: 88627, 9: 81050, 10: 56232}
prevalence per inspection: {'brakes': 0.0837, 'electrical (lighting, battery, wiring)': 0.1854, 'engine-related (emissions, fuel, mounts)': 0.0297, 'any defect': 0.4722}


          brakes  electrical  engine_emissions       n
fuel                                                  
diesel    0.0774      0.1809            0.0275   66527
electric  0.0227      0.0885            0.0020   71509
hybrid    0.0365      0.0952            0.0059   31382
lpg_cng   0.1230      0.2033            0.0508     610
petrol    0.0934      0.2011            0.0342  641993


## 2. Protocol
* Split **by plate** 70 / 15 / 15 so one car never appears in two parts (a car is inspected several times).
* **Scenarios** (features that exist for an arbitrary Israeli plate in the government registry: manufacture year → age, fuel, make, model name):
  * **G** = age, fuel, make · **G+model** = + model name.
  * **G+model+cc/mass** (diagnostic only): + engine size, kerb mass — fields the Israeli registry resource does NOT have, to show what a richer registry would add.
  * **H (future data)**: + defects found at the car's previous inspection. This is what the system would know only after it has recorded inspections itself, or if the Ministry shared them.
* Models: XGBoost on the natural objective (so probabilities stay probabilities), Logistic Regression baseline.
* Metrics: ROC-AUC, PR-AUC, top-20% capture, calibration error, Brier; baselines: prevalence and an age-only model.

In [4]:
d = d.sort_values(["kenteken", "date"])
for s in ["brakes", "electrical", "engine_emissions", "any_defect"]:
    d["prev_" + s] = d.groupby("kenteken")[s].shift(1)
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=0)
tr_i, tmp_i = next(gss.split(d, groups=d.kenteken)); tmp = d.iloc[tmp_i]
_, te_rel = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=0).split(tmp, groups=tmp.kenteken))
tr, te = d.iloc[tr_i].copy(), tmp.iloc[te_rel].copy()
assert not set(tr.kenteken) & set(te.kenteken)
for df in (tr, te):
    for c in ["fuel", "make", "model"]:
        df[c] = pd.Categorical(df[c], categories=sorted(d[c].unique()))
print(f"train {len(tr):,} ({tr.kenteken.nunique():,} cars) | test {len(te):,} ({te.kenteken.nunique():,} cars) | no plate in both parts")
SCEN = {"G (age, fuel, make)": ["age_year", "fuel", "make"], "G+model": ["age_year", "fuel", "make", "model"],
        "G+model+cc/mass (not in Israeli registry)": ["age_year", "fuel", "make", "model", "cilinderinhoud", "massa_rijklaar"],
        "G+model, same rows as H (reference)": ["age_year", "fuel", "make", "model", "SAMEROWS"],
        "H: + previous inspection result (future data)": ["age_year", "fuel", "make", "model", "PREV"]}
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop")
    tb = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(tb.p - tb.y) * tb.n).sum() / tb.n.sum()), tb
def capture(y, p, frac=0.2):
    k = int(len(p) * frac); idx = np.argsort(-p)[:k]; y = np.asarray(y); return float(y[idx].sum() / y.sum())
def fit_xgb(X, y, **kw):
    return XGBClassifier(n_estimators=250, max_depth=6, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, tree_method="hist",
                         enable_categorical=True, n_jobs=8, random_state=0, **kw).fit(X, y)

train 568,807 (259,874 cars) | test 121,479 (55,688 cars) | no plate in both parts


In [5]:
rows, store = [], {}
for tname, tcol in TARGETS.items():
    for sname, cols in SCEN.items():
        use = ["prev_" + tcol if c == "PREV" else c for c in cols if c != "SAMEROWS"]
        trs, tes = tr, te
        if "PREV" in cols or "SAMEROWS" in cols:                       # only inspections that have a previous one
            trs, tes = tr[tr["prev_" + tcol].notna()], te[te["prev_" + tcol].notna()]
        m = fit_xgb(trs[use], trs[tcol]); p = m.predict_proba(tes[use])[:, 1]; y = tes[tcol]
        e, _ = ece(y, p)
        rows.append({"system": tname, "features": sname, "n test": len(tes), "prevalence": y.mean(), "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
                     "capture in top 20%": capture(y, p), "calibration err": e, "Brier": brier_score_loss(y, p)})
        store[(tname, sname)] = (m, p, tes)
results = pd.DataFrame(rows)
display(results)

,system,features,n test,prevalence,ROC-AUC,PR-AUC,capture in top 20%,calibration err,Brier
0,brakes,"G (age, fuel, make)",121479,0.0842,0.6969,0.1601,0.4028,0.0018,0.0741
1,brakes,G+model,121479,0.0842,0.7021,0.1649,0.4138,0.0024,0.0739
2,brakes,G+model+cc/mass (not in Israeli registry),121479,0.0842,0.7042,0.1664,0.4190,0.0017,0.0739
3,brakes,"G+model, same rows as H (reference)",65791,0.0959,0.6743,0.1670,0.3777,0.0038,0.0839
4,brakes,H: + previous inspection result (future data),65791,0.0959,0.6846,0.1886,0.3932,0.0053,0.0832
5,"electrical (lighting, battery, wiring)","G (age, fuel, make)",121479,0.1860,0.6860,0.3001,0.3511,0.0040,0.1417
6,"electrical (lighting, battery, wiring)",G+model,121479,0.1860,0.6913,0.3065,0.3600,0.0041,0.1412
7,"electrical (lighting, battery, wiring)",G+model+cc/mass (not in Israeli registry),121479,0.1860,0.6929,0.3075,0.3613,0.0041,0.1410
8,"electrical (lighting, battery, wiring)","G+model, same rows as H (reference)",65791,0.2105,0.6738,0.3205,0.3336,0.0056,0.1562
9,"electrical (lighting, battery, wiring)",H: + previous inspection result (future data),65791,0.2105,0.6924,0.3599,0.3681,0.0066,0.1532


## 3. Logistic Regression baseline vs XGBoost (G+model scenario is too wide for LR; use G and age-only)

In [6]:
rows = []
for tname, tcol in TARGETS.items():
    y = te[tcol]
    for nm, num, ctg in [("age only", ["age_year"], []), ("age, fuel, make", ["age_year"], ["fuel", "make"])]:
        lr = Pipeline([("p", ColumnTransformer([("n", StandardScaler(), num)] + ([("c", OneHotEncoder(handle_unknown="ignore", min_frequency=200, sparse_output=False), ctg)] if ctg else []))), ("m", LogisticRegression(max_iter=1000))])
        lr.fit(tr[num + ctg].astype({c: str for c in ctg}), tr[tcol]); p = lr.predict_proba(te[num + ctg].astype({c: str for c in ctg}))[:, 1]
        rows.append({"system": tname, "model": "LogReg, " + nm, "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "capture in top 20%": capture(y, p), "calibration err": ece(y, p)[0]})
    p = store[(tname, "G (age, fuel, make)")][1]
    rows.append({"system": tname, "model": "XGBoost, age, fuel, make", "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "capture in top 20%": capture(y, p), "calibration err": ece(y, p)[0]})
display(pd.DataFrame(rows))

,system,model,ROC-AUC,PR-AUC,capture in top 20%,calibration err
0,brakes,"LogReg, age only",0.6840,0.1428,0.3798,0.0110
1,brakes,"LogReg, age, fuel, make",0.6908,0.1504,0.3951,0.0082
2,brakes,"XGBoost, age, fuel, make",0.6969,0.1601,0.4028,0.0018
3,"electrical (lighting, battery, wiring)","LogReg, age only",0.6645,0.2753,0.3323,0.0204
4,"electrical (lighting, battery, wiring)","LogReg, age, fuel, make",0.6812,0.2912,0.3480,0.0193
5,"electrical (lighting, battery, wiring)","XGBoost, age, fuel, make",0.6860,0.3001,0.3511,0.0040
6,"engine-related (emissions, fuel, mounts)","LogReg, age only",0.7620,0.0685,0.4790,0.0070
7,"engine-related (emissions, fuel, mounts)","LogReg, age, fuel, make",0.7748,0.0771,0.5286,0.0053
8,"engine-related (emissions, fuel, mounts)","XGBoost, age, fuel, make",0.7875,0.0818,0.5481,0.0015
9,any defect,"LogReg, age only",0.6715,0.6179,0.2879,0.0197


## 4. Is the signal stable and are the probabilities usable? Reliability table for brakes (G+model) and bootstrap interval of the AUC by plate

In [7]:
m, p, tes = store[("brakes", "G+model")]
e, tb = ece(tes["brakes"], p); print(f"expected calibration error {e:.4f}"); display(tb.rename(columns={"p": "mean predicted", "y": "observed rate"}))
rng = np.random.default_rng(0); plates = tes.kenteken.unique(); idx_by = tes.reset_index(drop=True).groupby("kenteken").indices; yv = tes["brakes"].to_numpy()
aucs = []
for _ in range(100):
    pick = rng.choice(plates, len(plates)); ii = np.concatenate([idx_by[k] for k in pick]); aucs.append(roc_auc_score(yv[ii], p[ii]))
print("brakes AUC, 95% bootstrap interval over cars:", np.percentile(aucs, [2.5, 97.5]).round(3))
imp = pd.Series(m.feature_importances_, index=SCEN["G+model"]).sort_values(ascending=False); print("feature importance:", imp.round(3).to_dict())

expected calibration error 0.0024


,mean predicted,observed rate,n
"(-0.0006540000000000001, 0.0211]",0.0117,0.0132,12263
"(0.0211, 0.0361]",0.0292,0.0326,12044
"(0.0361, 0.0455]",0.0409,0.0408,12415
"(0.0455, 0.0565]",0.0508,0.0484,12045
"(0.0565, 0.0661]",0.0611,0.0620,11980
"(0.0661, 0.0824]",0.0738,0.0728,12300
"(0.0824, 0.107]",0.0943,0.0972,12002
"(0.107, 0.137]",0.1224,0.1273,12151
"(0.137, 0.172]",0.1537,0.1489,12138
"(0.172, 0.49]",0.2015,0.1995,12141


brakes AUC, 95% bootstrap interval over cars: [0.697 0.707]
feature importance: {'age_year': 0.47200000286102295, 'model': 0.23199999332427979, 'fuel': 0.19099999964237213, 'make': 0.10499999672174454}


## 5. Does it hold when age is the only thing Stage 1 can pass on?
In Israel the Stage-1 mileage estimate is a function of age, fuel and (after calibration) ownership only. A tree model that already has age and fuel gains **nothing** from such an estimate:
it is a monotone re-coding of variables it has. The MOT pilot (notebook 04) tests this directly with a trained Stage-1 estimate versus the true odometer.